In [1]:
import sys
sys.path.append('..')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset

from src.models.transformers import MODEL_REGISTRY
from src.utils.parser import Config
from src.utils.text_gen import simple_text_gen

/home/leo/.python-envs/general-3.10.12-env/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
import tiktoken
tokenizer = tiktoken.get_encoding('gpt2')

In [3]:
ckpnt_dir = '../logs/GPT2/v1'
config = Config.from_yaml(ckpnt_dir + '/hparams.yaml')

model = MODEL_REGISTRY[config.model_params.pop("model_name")](**config.model_params).to(torch.device('cpu'))

ckpnt = torch.load(ckpnt_dir + '/checkpoints/step_100200.pt', weights_only=False)
# Remove `_orig_mod.` from keys
model_state_dict = {}
for k, v in ckpnt["model"].items():
    new_key = k.replace("_orig_mod.", "")  # Strip the prefix
    model_state_dict[new_key] = v


model.load_state_dict(model_state_dict)
model.to(torch.device('cuda:0'))
model = model.eval()

In [5]:
print(
    *simple_text_gen(
        'The capital of Italy is',
        model,
        tokenizer,
        topk=50,
        max_len_return_seq=25,
        n_return_seq=5
    ),
    sep='\n'
)

> The capital of Italy is about on not on other has with the its these, a-. not also has a aboutThe
> The capital of Italy is of the
 to or by are into, than and two such on " the also were many their
> The capital of Italy is been., where the alsoThe with to as not a as on are, into are as other
> The capital of Italy isThe with is their in at a- some but such ( an's for to not,. is
> The capital of Italy is- in not,, not has many or one of on which
 or be and., was
